# Humanities researchers on research data management — walkthrough

**One row is** one survey response from a humanities researcher.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-datasets/blob/main/datasets/01_humanities_rdm_survey/walkthrough.ipynb)

**This notebook is written to be read, not filled in.** Every cell already works. Run them in order, read the explanation above each one, and look at what comes out.

When you reach the end, go back and change one thing — a column name, a number — and run it again. You cannot break the data: it is a separate file and this notebook only reads it.

---

## Step 0 — Load the data

`pd.read_csv` reads the file into a table. We call the table `df`, which is just a name — it is short for *dataframe*.

In Colab you will be asked to upload `data.csv` first.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('data.csv'):
    try:
        from google.colab import files
        print('Upload data.csv from your dataset folder')
        files.upload()
    except ImportError:
        raise SystemExit('Place data.csv next to this notebook.')

df = pd.read_csv('data.csv')
print(f'{len(df):,} rows and {df.shape[1]} columns')

## Step 1 — Look at it

`.head()` shows the first five rows. Always look before you calculate: you will often spot a problem here that no statistic would tell you about.

In [ ]:
df.head()

## Step 2 — The absence audit

Which columns are incomplete? `.isna()` marks every empty cell, and `.mean()` turns that into a proportion.

This is the same thing as `=COUNTBLANK` in Excel, and it is worth **30% of the assignment mark**. A column that is mostly empty cannot support a confident claim.

In [ ]:
pd.DataFrame({
    'type': df.dtypes.astype(str),
    'missing_%': (df.isna().mean() * 100).round(1),
    'distinct': df.nunique(),
}).sort_values('missing_%', ascending=False)

## Step 3 — Describe one column

`in_which_humanities_discipline_is_your_research_based` is a **categorical** column — a label, not a quantity. The useful summary is a count of each value.

`value_counts()` does that. In Excel this is a pivot table with `in_which_humanities_discipline_is_your_research_based` in Rows and Count as the value.

In [ ]:
counts = df['in_which_humanities_discipline_is_your_research_based'].value_counts()
print(f'{df["in_which_humanities_discipline_is_your_research_based"].nunique():,} distinct values')
counts.head(10)

## Step 4 — Compare groups with a cross-tabulation

This dataset has no quantity to average — every column is a label. That is normal for survey data, and the right tool is a **cross-tabulation**: count how often each combination of two answers occurs.

Here we cross `in_which_humanities_discipline_is_your_research_based` against `in_what_region_are_you_based`. In Excel this is a pivot table with one column in Rows and the other in Columns.

In [ ]:
ct = pd.crosstab(df['in_which_humanities_discipline_is_your_research_based'], df['in_what_region_are_you_based'])
ct

### Counts mislead — use percentages

A big group produces big numbers in every cell, so raw counts mostly tell you which group is big. Convert to **row percentages** and you are comparing like with like.

This is the trap named in your brief: always check the denominator.

In [ ]:
pct = pd.crosstab(df['in_which_humanities_discipline_is_your_research_based'], df['in_what_region_are_you_based'], normalize='index') * 100
pct.round(1)

## Step 5 — Could the pattern be chance?

For two categorical columns the test is a **chi-squared test**. It asks whether the combinations occur more or less often than you would expect if the two answers were unrelated.

In [ ]:
from scipy import stats

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f'chi-squared : {chi2:,.2f}')
print(f'p-value     : {p:.4g}')
print(f'smallest expected count: {expected.min():.1f}')
print()
print('p below 0.05 means the two answers are related.')
print('If any expected count is below 5, the test is unreliable —')
print('combine small categories first.')

## Step 6 — One chart

A grouped bar chart of the percentages. Percentages, not counts, so groups of different sizes can be compared.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pct.head(8).plot(kind='barh', ax=ax)
ax.set_xlabel('% within row')
ax.set_title('in_what_region_are_you_based by in_which_humanities_discipline_is_your_research_based (row percentages)')
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Step 7 — What this does not show

Almost every column is categorical, so it is tempting to report percentages of the whole sample. Check the denominator for each question first: many were shown only to a subset, so '40% said yes' may be 40% of 90 people, not of 371.

Write your limits section before you write your finding. It is easier, it is worth more marks, and it stops you claiming something the data cannot support.

---

## Now change something

Go back and edit one thing, then run the cell again:

- swap `in_which_humanities_discipline_is_your_research_based` for a different categorical column
- swap `None` for a different numeric column
- change `head(10)` to `head(20)`
- change `>= 5` to `>= 20` and watch small groups disappear

If you get a red error, read the **last** line — it names the problem. `KeyError` almost always means a column name is spelled differently. Reload the notebook and you are back where you started.